# Guorong Ji and Brandon Chen
# FRE-6103 MP2

# Step 1
Create a pandas dataframe from the Excel file given.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime

def prep_yield_curve_data(filepath):
    # 1. Load the data, skipping the first 7 rows to make Row 8 the header
    df = pd.read_excel(filepath, skiprows=7)

    # 2. Explicitly grab only columns 1 through 6 (Maturity through Asked Yield)
    # This ignores any stray text notes left in the far right columns of the spreadsheet
    df = df.iloc[:, 1:7].copy()

    # 3. Standardize column names for easier indexing
    df.columns = ['Maturity', 'Coupon', 'Bid', 'Asked', 'Chg', 'Asked_Yield']

    # 4. Cast data types to ensure accurate math operations
    df['Maturity'] = pd.to_datetime(df['Maturity'], errors='coerce')
    numeric_cols = ['Coupon', 'Bid', 'Asked', 'Chg', 'Asked_Yield']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop any rows that failed to parse (e.g., empty rows at the bottom of the sheet)
    df = df.dropna(subset=['Maturity', 'Bid', 'Asked'])

    # 5. Calculate the Mid Price (P)
    df['Mid_Price'] = (df['Bid'] + df['Asked']) / 2

    # 6. Calculate Time to Maturity (t) in years
    # (Using September 4, 2026, as the settlement date)
    settlement_date = pd.to_datetime('2026-09-04')
    df['Time_to_Maturity'] = (df['Maturity'] - settlement_date).dt.days / 365.0

    # 7. Handle Duplicate Maturities (Find most liquid bonds)
    df['Distance_to_Par'] = abs(df['Mid_Price'] - 100)
    idx_closest_to_par = df.groupby('Maturity')['Distance_to_Par'].idxmin()
    df_clean = df.loc[idx_closest_to_par].copy()

    # 8. Sort chronologically for bootstrapping and clean up
    df_clean = df_clean.sort_values(by='Time_to_Maturity').reset_index(drop=True)
    df_clean = df_clean.drop(columns=['Distance_to_Par'])

    return df_clean

# Execute the transformation
treasury_df = prep_yield_curve_data('data/Treasury data 090426.xlsx')
print(treasury_df.head())

    Maturity  Coupon      Bid    Asked    Chg  Asked_Yield  Mid_Price  \
0 2026-09-15   4.625  100.000  100.010 -0.002        2.937    100.005   
1 2026-09-30   3.500   99.306   99.316  0.002        3.603     99.311   
2 2026-10-15   4.625  100.022  100.032 -0.002        3.576    100.027   
3 2026-10-31   4.125  100.000  100.010 -0.006        3.877    100.005   
4 2026-11-15   4.625  100.040  100.050 -0.002        3.741    100.045   

   Time_to_Maturity  
0          0.030137  
1          0.071233  
2          0.112329  
3          0.156164  
4          0.197260  


# Methodology
